In [53]:
import torch
torch.cuda.empty_cache()

In [54]:
torch.cuda.ipc_collect()

In [55]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [56]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [57]:
!kill -9 626

/bin/bash: line 1: kill: (626) - No such process


In [58]:
torch.cuda.empty_cache()

In [59]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from timm import create_model

import numpy as np
import cv2
import gc
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

In [60]:
# 🔥 CLAHE function
id="fixed_clahe"
from PIL import Image
import numpy as np
import cv2

class CLAHETransform:
    def __call__(self, img):
        img = np.array(img)  # PIL → numpy

        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img = clahe.apply(img)

        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        return Image.fromarray(img)  # 🔥 FIX: numpy → PIL


train_transform = transforms.Compose([
    CLAHETransform(),  # 🔥 NEW
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=10, translate=(0.05,0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    CLAHETransform(),  # 🔥 NEW
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [61]:
data_dir = "/kaggle/input/datasets/venkatsaikondra/multi-venkat/Final_Data"

train_dataset = datasets.ImageFolder(data_dir + "/train", transform=train_transform)
val_dataset = datasets.ImageFolder(data_dir + "/val", transform=val_transform)
test_dataset = datasets.ImageFolder(data_dir + "/test", transform=val_transform)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=False)

In [62]:
class ResViT(nn.Module):
    def __init__(self, num_classes=4):
        super(ResViT, self).__init__()

        # CNN
        self.cnn = models.resnet18(weights="IMAGENET1K_V1")
        cnn_dim = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity()

        # ViT
        self.vit = create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
        vit_dim = 384

        self.feature_dim = cnn_dim + vit_dim

        # 🔥 Gated Fusion
        self.gate = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.feature_dim),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f_cnn = self.cnn(x)
        f_vit = self.vit(x)

        # 🔥 Normalize features (IMPORTANT)
        f_cnn = nn.functional.normalize(f_cnn, dim=1)
        f_vit = nn.functional.normalize(f_vit, dim=1)

        f = torch.cat([f_cnn, f_vit], dim=1)

        g = self.gate(f)
        f = f * g

        return self.classifier(f)

In [63]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ResViT().to(device)

# 🔥 Weighted Loss (fix class confusion)
class_weights = torch.tensor([1.0, 1.0, 1.3, 1.3]).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

scaler = GradScaler()

/tmp/ipykernel_55/229564915.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [64]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return correct / total

In [65]:
def train_model(model, train_loader, val_loader, epochs=20):
    best_acc = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in tqdm(train_loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            del outputs, loss

        scheduler.step()

        val_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

    return model

In [66]:
model = train_model(model, train_loader, val_loader, epochs=25)

print("\nFinal Test Performance:")
test_acc = evaluate(model, test_loader)
print(f"Final Test Accuracy: {test_acc:.4f}")

  0%|          | 0/944 [00:00<?, ?it/s]/tmp/ipykernel_55/4189186203.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 944/944 [01:11<00:00, 13.21it/s]



Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       404
           1       0.93      0.93      0.93       404
           2       0.61      0.95      0.74       404
           3       0.87      0.40      0.54       404

    accuracy                           0.81      1616
   macro avg       0.85      0.81      0.80      1616
weighted avg       0.85      0.81      0.80      1616


Confusion Matrix:
[[391   0  10   3]
 [  2 377  19   6]
 [  0   6 383  15]
 [  1  22 221 160]]
Epoch 1, Loss: 793.8297, Val Acc: 0.8113


  0%|          | 0/944 [00:00<?, ?it/s]/tmp/ipykernel_55/4189186203.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 944/944 [01:10<00:00, 13.37it/s]



Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       404
           1       0.96      0.93      0.94       404
           2       0.65      0.94      0.77       404
           3       0.88      0.50      0.64       404

    accuracy                           0.84      1616
   macro avg       0.87      0.84      0.84      1616
weighted avg       0.87      0.84      0.84      1616


Confusion Matrix:
[[403   0   0   1]
 [  1 374  19  10]
 [  3   3 381  17]
 [  1  11 188 204]]
Epoch 2, Loss: 649.3105, Val Acc: 0.8428


  0%|          | 0/944 [00:00<?, ?it/s]/tmp/ipykernel_55/4189186203.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 944/944 [01:11<00:00, 13.16it/s]


...


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       404
           1       0.97      0.98      0.98       404
           2       0.86      0.83      0.84       404
           3       0.83      0.86      0.84       404

    accuracy                           0.91      1616
   macro avg       0.91      0.91      0.91      1616
weighted avg       0.91      0.91      0.91      1616


Confusion Matrix:
[[400   2   0   2]
 [  0 395   3   6]
 [  0   5 335  64]
 [  0   4  53 347]]
Epoch 23, Loss: 448.7091, Val Acc: 0.9140


  0%|          | 0/944 [00:00<?, ?it/s]/tmp/ipykernel_55/4189186203.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 944/944 [01:08<00:00, 13.81it/s]



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.96      0.99      0.97       404
           2       0.86      0.80      0.83       404
           3       0.82      0.86      0.84       404

    accuracy                           0.91      1616
   macro avg       0.91      0.91      0.91      1616
weighted avg       0.91      0.91      0.91      1616


Confusion Matrix:
[[402   1   0   1]
 [  0 398   3   3]
 [  0   8 322  74]
 [  0   9  49 346]]
Epoch 24, Loss: 446.1134, Val Acc: 0.9084


  0%|          | 0/944 [00:00<?, ?it/s]/tmp/ipykernel_55/4189186203.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 944/944 [01:08<00:00, 13.80it/s]



Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       404
           1       0.99      0.97      0.98       404
           2       0.80      0.86      0.83       404
           3       0.84      0.80      0.82       404

    accuracy                           0.91      1616
   macro avg       0.91      0.91      0.91      1616
weighted avg       0.91      0.91      0.91      1616


Confusion Matrix:
[[401   0   1   2]
 [  0 391   7   6]
 [  0   3 348  53]
 [  0   2  79 323]]
Epoch 25, Loss: 440.4266, Val Acc: 0.9053

Final Test Performance:

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       405
           1       0.98      0.95      0.97       405
           2       0.80      0.88      0.84       405
           3       0.85      0.79      0.82       405

    accuracy                           0.90      1620
   macro avg       0.91     